In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = BASE_DIR / "data" / "raw"
SYNTHETIC_DIR = BASE_DIR / "data" / "synthetic"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUTPUTS_DIR = BASE_DIR / "outputs"

SYNTHETIC_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(BASE_DIR)

In [ ]:
dates = pd.date_range(start="2025-01-01", end="2025-12-31", freq="D")

df = pd.DataFrame({
    "date": dates
})

df.head()

In [ ]:
df["blower_id"] = "ZG_150_Blower_01"
df["site_id"] = "Site_01"
df["operational_class"] = "High Duty"

df.head()

In [ ]:
df["max_op_ambient_temp_c"] = 48.9   # 120°F converted to °C
df["min_op_ambient_temp_c"] = -15  # -40°F converted to °C
df["max_op_discharge_temp_c"] = 167.2  # 333°F converted to °C
df["max_op_pressure_diff_psi"] = 15
df["service_interval_days"] = 365
df["max_op_rpm"] = 3000

In [ ]:
rng = np.random.default_rng(42)

df["daily_op_hours"] = np.random.default_rng(42).normal(loc=21, scale=1.5, size=len(df))
df["daily_op_hours"] = df["daily_op_hours"].clip(20, 24)

df["daily_load_percent"] = np.random.default_rng(42).normal(loc=85, scale=7, size=len(df))
df["daily_load_percent"] = df["daily_load_percent"].clip(70, 100)

df["rpm"] = df["max_op_rpm"] * (df["daily_load_percent"] / 100)
df.head()

In [ ]:
df["cumulative_op_hours"] = df["daily_op_hours"].cumsum()

df[[
    "date",
    "daily_op_hours",
    "cumulative_op_hours"
]].head()

In [ ]:
weather_path = RAW_DIR / "Giyani data.csv"

weather_df = pd.read_csv(weather_path)

weather_df.head()

In [ ]:
weather_df["datetime"] = pd.to_datetime(weather_df["datetime"], format="%Y-%m-%d")

In [ ]:
weather_df = weather_df.rename(columns={
    "name": "location_name",
    "datetime": "date",
    "tempmax": "temp_max_F",
    "tempmin": "temp_min_F",
    "temp": "temp_avg_F",
    "windgust": "wind_gust_kph",
})

weather_df.head()

In [ ]:
#Dropping unnecessary columns and converting temperature from Fahrenheit to Celsius
from operator import index


weather_df = weather_df[["date", "temp_max_F", "temp_min_F", "temp_avg_F","humidity", "wind_gust_kph"]]
weather_df["amb_temp_max_c"] = (weather_df["temp_max_F"] - 32) * 5.0/9.0
weather_df["amb_temp_min_c"] = (weather_df["temp_min_F"] - 32) * 5.0/9.0
weather_df["amb_temp_avg_c"] = (weather_df["temp_avg_F"] - 32) * 5.0/9.0
weather_df = weather_df[["date", "amb_temp_max_c", "amb_temp_min_c", "amb_temp_avg_c","humidity", "wind_gust_kph"]]

weather_df.head()
#save the cleaned weather data for future use
weather_df.to_csv(PROCESSED_DIR/ "processed_weather_data.csv", index =False)

In [ ]:
#merging weather data into the main dataset
df = df.merge(weather_df, on="date", how="left")
df.head()

In [ ]:
#dust index is a synthetic feature that takes into account wind gusts and humidity to estimate the potential for dust accumulation on the blower, which can impact its performance and maintenance needs.
# Normalising wind contribution
wind_component = (
    df["wind_gust_kph"] / df["wind_gust_kph"].max()
)

# Inverse humidity contribution (use existing 'humidity' column)
humidity_component = 1 - (
    df["humidity"] / 100
)

# Combined dust index 
df["dust_index_raw"] = (
    0.6 * wind_component
    + 0.4 * humidity_component
    + rng.normal(loc=0, scale=0.05, size=len(df))
)

# Scale to 0–100
df["dust_index"] = (df["dust_index_raw"] * 100).clip(0, 100)

# Round values
df["dust_index"] = df["dust_index"].round(2)

df[[
    "date",
    "wind_gust_kph",
    "humidity",
    "dust_index"
]].head()

In [ ]:
#Simulating synthetic pressure differential feature that estimates the difference in pressure across the blower, 
#which can indicate potential blockages or performance issues. It is calculated based on the blower's load and dust_index, as these factors can influence the pressure conditions.
df["pressure_diff_psi"] = (
    4
    + 0.09 * df["daily_load_percent"]
    + 0.04 * df["dust_index"]
    + rng.normal(0, 0.5, len(df))
)

df["pressure_diff_psi"] = df["pressure_diff_psi"].clip(2, 15)

df[[
    "daily_load_percent",
    "dust_index",
    "pressure_diff_psi"
]].head()


In [ ]:
#simulating machine operating temperature
df["casing_temperature_c"] = (
    df["amb_temp_avg_c"]
    + 0.12 * df["daily_load_percent"]
    + 0.6 * df["pressure_diff_psi"]
    + rng.normal(loc=0, scale=1.5, size=len(df))
)

# Keep realistic operational range
df["casing_temperature_c"] = df["casing_temperature_c"].clip(25, 120)

df[[
    "amb_temp_avg_c",
    "daily_load_percent",
    "pressure_diff_psi",
    "casing_temperature_c"
]].head()

In [ ]:
#simulating machine vibration level
df["vibration_mm_s"] = (
    1.5
    + 0.025 * df["daily_load_percent"]
    + 0.12 * df["pressure_diff_psi"]
    + 0.00012 * df["cumulative_op_hours"]
    + rng.normal(loc=0, scale=0.3, size=len(df))
)

df["vibration_mm_s"] = df["vibration_mm_s"].clip(1, 12)

df[[
    "daily_load_percent",
    "pressure_diff_psi",
    "vibration_mm_s"
]].head()

In [ ]:
#Calculating degradation index as a composite score that estimates the overall health of the blower based on key operational parameters. 
rng = np.random.default_rng(42)

temp_weight = rng.uniform(0.000035, 0.000050)
vibration_weight = rng.uniform(0.00045, 0.00065)
pressure_weight = rng.uniform(0.00002, 0.00003)

daily_degradation = (
    temp_weight * df["casing_temperature_c"]
    + vibration_weight * df["vibration_mm_s"]
    + pressure_weight * df["pressure_diff_psi"]
    + rng.normal(loc=0, scale=0.0003, size=len(df))
)
daily_degradation = daily_degradation.clip(0)

df["daily_degradation"] = daily_degradation

maintenance_threshold = 0.60
failure_threshold = 1.0

degradation_values = []
maintenance_events = []
adjusted_daily_degradation = []

current_degradation = 0

for daily_deg in df["daily_degradation"]:

    degradation_multiplier = (
        1 + (current_degradation / failure_threshold) ** 2
    )

    daily_deg = daily_deg * degradation_multiplier

    adjusted_daily_degradation.append(daily_deg)

    current_degradation += daily_deg

    if current_degradation >= (
        maintenance_threshold * failure_threshold
    ):

        maintenance_events.append(1)

        # imperfect maintenance recovery
        current_degradation = current_degradation * 0.20

    else:
        maintenance_events.append(0)

    degradation_values.append(current_degradation)

df["daily_degradation"] = adjusted_daily_degradation

df["degradation_index"] = degradation_values

df["maintenance_event"] = maintenance_events

df["health_score"] = (
    100 * (
        1 - (
            df["degradation_index"]
            / failure_threshold
        )
    )
).clip(0, 100)

average_daily_degradation = df["daily_degradation"].mean()

df["RUL_days"] = (
    (
        failure_threshold
        - df["degradation_index"]
    )
    / average_daily_degradation
).clip(lower=0)

df[[
    "casing_temperature_c",
    "vibration_mm_s",
    "daily_degradation",
    "degradation_index",
    "maintenance_event",
    "health_score",
    "RUL_days"
]].tail()

In [ ]:
df["failure_threshold"] = failure_threshold

In [ ]:
#defining failure event based on degradation index exceeding the threshold
df["failure_event"] = (df["degradation_index"] >= df["failure_threshold"]).astype(int)
df[["date","degradation_index", "failure_threshold", "failure_event"]].sample(15)

In [ ]:
#defining maintenance event triggered label
maintenance_threshold = 0.60
df["maintenance_event_triggered"] = (df["degradation_index"] >= (maintenance_threshold * df["failure_threshold"])).astype(int)
df[["date","degradation_index", "failure_threshold", "maintenance_event_triggered"]].sample(25)

In [ ]:
#calculating remaining useful life (RUL) as the number of days until the degradation index reaches the failure threshold, providing a time-based estimate of when maintenance or replacement will be necessary.
df["daily_degradation_rate"] = df["degradation_index"].diff().fillna(0)
average_degradation_rate = df["daily_degradation_rate"].mean()
df["RUL_days"] = ((df["failure_threshold"] - df["degradation_index"]) / average_degradation_rate).clip(lower=0)
df[["date", "degradation_index", "failure_threshold", "RUL_days"]].sample(20)

In [ ]:
df[["date","daily_op_hours", "daily_load_percent", "casing_temperature_c", "vibration_mm_s", "pressure_diff_psi", "degradation_index", "health_score", "failure_event", "maintenance_event_triggered", "RUL_days"]].head()

In [ ]:
#visualising degraddation trend
plt.figure(figsize=(12, 5)) 
plt.plot(df["date"], df["degradation_index"], label="Degradation Index", color="blue")
plt.axhline(y=failure_threshold, color="red", linestyle="--", label="Failure Threshold")
plt.title("Blower Degradation Index Over Time")
plt.xlabel("Date")
plt.ylabel("Degradation Index")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(df["date"], df["health_score"])
plt.title("Health Score Over Time")
plt.xlabel("Date")
plt.ylabel("Health Score")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(df["date"], df["RUL_days"])
plt.title("Remaining Useful Life (RUL)")
plt.xlabel("Date")
plt.ylabel("Days")
plt.grid(True)
plt.show()

In [ ]:
output_path = SYNTHETIC_DIR / "baseline_single_blower_2025.csv"

df.to_csv(output_path, index=False)

print(f"Dataset saved to: {output_path}")

In [ ]:
df = pd.read_csv(output_path)
print(df[["date","daily_op_hours","amb_temp_avg_c","amb_temp_min_c","amb_temp_max_c","daily_load_percent", "casing_temperature_c", "vibration_mm_s", "pressure_diff_psi", "degradation_index", "health_score", "failure_event", "maintenance_event_triggered", "RUL_days"]].describe())

In [ ]:
df[df["maintenance_event_triggered"] == 1][[
    "date",
    "degradation_index",
    "health_score",
    "RUL_days"
]].head()

- One rotary blower was simulated for daily operation across 2025.
- Real Giyani weather data was used for ambient temperature, humidity and wind gust.
- Operational load and operating hours were synthetically generated using a high-duty profile.
- Pressure differential was modelled as a function of load and dust exposure.
- Casing temperature was modelled as a function of ambient temperature, load and pressure differential.
- Vibration was modelled as a function of load, pressure differential and cumulative operating hours.
- Degradation was modelled cumulatively using casing temperature, vibration and pressure differential.
- Maintenance is triggered when degradation reaches 75% of the failure threshold.

#Multi blower fleet simulation

In [ ]:
operational_profiles = {
    "Low Duty": {"daily_op_hours_mean": 9, "daily_op_hours_std": 1.5,"daily_op_hours_min": 6, "daily_op_hours_max": 12,"load_mean": 50, "load_std": 6, "load_min": 35, "load_max": 65},
    "Medium Duty": {"daily_op_hours_mean": 16, "daily_op_hours_std": 2,"daily_op_hours_min": 13, "daily_op_hours_max": 19,"load_mean": 70, "load_std": 7, "load_min": 55, "load_max": 85},
    "High Duty": {"daily_op_hours_mean": 22, "daily_op_hours_std": 1.5,"daily_op_hours_min": 20, "daily_op_hours_max": 24,"load_mean": 85, "load_std": 7, "load_min": 70, "load_max": 100}
}

fleet_config = [
    {"blower_id": "ZG150_B001", "site_id": "Site_01", "operational_class": "High Duty","stress_factor": 1.14},
    {"blower_id": "ZG150_B002", "site_id": "Site_01", "operational_class": "High Duty"},
    {"blower_id": "ZG150_B003", "site_id": "Site_01", "operational_class": "Medium Duty","stress_factor": 1.1},
    {"blower_id": "ZG150_B004", "site_id": "Site_01", "operational_class": "Medium Duty"},

    {"blower_id": "ZG200_B011", "site_id": "Site_02", "operational_class": "High Duty"},
    {"blower_id": "ZG200_B012", "site_id": "Site_02", "operational_class": "Medium Duty"},
    {"blower_id": "ZG200_B013", "site_id": "Site_02", "operational_class": "Low Duty", "stress_factor": 1.25},
    {"blower_id": "ZG200_B014", "site_id": "Site_02", "operational_class": "Low Duty"},

    {"blower_id": "ZG250_B021", "site_id": "Site_03", "operational_class": "High Duty"},
    {"blower_id": "ZG250_B022", "site_id": "Site_03", "operational_class": "Medium Duty"},
    {"blower_id": "ZG250_B023", "site_id": "Site_03", "operational_class": "Medium Duty"},
    {"blower_id": "ZG250_B024", "site_id": "Site_03", "operational_class": "Low Duty", "stress_factor": 1.2},
]


In [ ]:
#simulating a fleet of blowers with different operational profiles to create a more complex and realistic dataset that can be used to test the robustness of predictive maintenance models across varying conditions.
def simulate_blower_data(blower_config,weather_df,seed=42):
    rng = np.random.default_rng(seed)
    profile = operational_profiles[blower_config["operational_class"]]
    blower_df = weather_df.copy()

    blower_df["blower_id"] = blower_config["blower_id"]
    blower_df["site_id"] = blower_config["site_id"] 
    blower_df["operational_class"] = blower_config["operational_class"] 

    blower_df["max_op_ambient_temp_c"] = 48.9
    blower_df["failure_threshold"] = 1.0
    blower_df["max_op_rpm"] = 3000
    blower_df["daily_op_hours"] = rng.normal(
        loc=profile["daily_op_hours_mean"],
        scale=profile["daily_op_hours_std"],
        size=len(blower_df)
    ).clip(profile["daily_op_hours_min"], profile["daily_op_hours_max"])    
    blower_df["cumulative_op_hours"] = blower_df["daily_op_hours"].cumsum()
    blower_df["daily_load_percent"] = rng.normal(
        loc=profile["load_mean"],
        scale=profile["load_std"],
        size=len(blower_df)
    ).clip(profile["load_min"], profile["load_max"])
    blower_df["rpm"] = blower_df["max_op_rpm"] * (blower_df["daily_load_percent"] / 100)
    wind_component = blower_df["wind_gust_kph"] / blower_df["wind_gust_kph"].max()
    humidity_component = 1 - (blower_df["humidity"] / 100)
    blower_df["dust_index"] = (0.6 * wind_component + 0.4 * humidity_component + rng.normal(loc=0, scale=0.05, size=len(blower_df))) * 100
    blower_df["dust_index"] = blower_df["dust_index"].clip(0, 100).round(2)
    blower_df["pressure_diff_psi"] = (4 + 0.08 * blower_df["daily_load_percent"] + 0.03 * blower_df["dust_index"] + rng.normal(loc=0, scale=0.5, size=len(blower_df))).clip(2, 15)
    blower_df["casing_temperature_c"] = blower_df["amb_temp_avg_c"] + 0.12 * blower_df["daily_load_percent"] + 0.6 * blower_df["pressure_diff_psi"] + rng.normal(loc=0, scale=1.5, size=len(blower_df))
    blower_df["casing_temperature_c"] = blower_df["casing_temperature_c"].clip(25, 120)

    blower_df["vibration_mm_s"] = (1.5 + 0.025 * blower_df["daily_load_percent"]+ 0.12 * blower_df["pressure_diff_psi"]+ 0.00008 * blower_df["cumulative_op_hours"]+ rng.normal(loc=0, scale=0.3, size=len(blower_df)))   
    blower_df["vibration_mm_s"] = blower_df["vibration_mm_s"].clip(1, 12)

    unit_variation = rng.normal(loc=1.0, scale=0.10, size=len(blower_df))

    stress_factor = blower_config.get("stress_factor", 1.0) 

    temp_weight = rng.uniform(0.000035, 0.000050)
    vibration_weight = rng.uniform(0.00045, 0.00065)
    pressure_weight = rng.uniform(0.00002, 0.00003)

    daily_degradation = (
    0.0002
    + temp_weight * blower_df["casing_temperature_c"]
    + vibration_weight * blower_df["vibration_mm_s"]
    + pressure_weight * blower_df["pressure_diff_psi"]
    + rng.normal(loc=0, scale=0.0003, size=len(blower_df))
    )

    daily_degradation = (
        daily_degradation
        * unit_variation
        * stress_factor
    ).clip(0)

    blower_df["daily_degradation"] = daily_degradation

    maintenance_threshold = 0.60
    failure_threshold = 1.0

    unit_maintenance_threshold = rng.normal(maintenance_threshold, 0.05)
    unit_maintenance_threshold = np.clip(unit_maintenance_threshold,0.60, 0.82)

    degradation_values = []
    maintenance_events = []
    adjusted_daily_degradation = []

    current_degradation = 0
    post_maintenance_factor = 1.0

    for daily_deg in blower_df["daily_degradation"]:

        degradation_multiplier = 1 + (current_degradation / failure_threshold) ** 2

        daily_deg = (daily_deg * degradation_multiplier* post_maintenance_factor)

        adjusted_daily_degradation.append(daily_deg)

        current_degradation += daily_deg

        if current_degradation >= unit_maintenance_threshold * failure_threshold:
         maintenance_events.append(1)

        # Maintenance partially restores condition
        #  reduces accumulated degradation by 80%
         current_degradation = current_degradation * 0.2
         post_maintenance_factor = 0.85
        else:
         maintenance_events.append(0)
         post_maintenance_factor = min(1.0, post_maintenance_factor + 0.002)

        degradation_values.append(current_degradation)
    blower_df["daily_degradation"] = adjusted_daily_degradation
    blower_df["degradation_index"] = degradation_values
    blower_df["maintenance_event"] = maintenance_events
    blower_df["maintenance_threshold"] = unit_maintenance_threshold

    blower_df["health_score"] = (100 * (1 - (blower_df["degradation_index"] / blower_df["failure_threshold"]))).clip(0, 100)

    blower_df["maintenance_event_triggered"] = (blower_df["maintenance_event"] == 1).astype(int)

    blower_df["failure_event"] = (blower_df["degradation_index"] >= blower_df["failure_threshold"]).astype(int)

    average_daily_degradation = blower_df["daily_degradation"].mean()

    blower_df["RUL_days"] = ((blower_df["failure_threshold"] - blower_df["degradation_index"]) / average_daily_degradation).clip(lower=0)

    return blower_df

In [39]:
#generate data for the entire fleet
fleet_data = []

for i, blower in enumerate(fleet_config):
    blower_data = simulate_blower_data(
        blower_config =blower,
        weather_df =weather_df,
        seed=42+i
    )
    fleet_data.append(blower_data)

fleet_df = pd.concat(fleet_data, ignore_index=True)
fleet_df.head()

,date,amb_temp_max_c,amb_temp_min_c,amb_temp_avg_c,humidity,wind_gust_kph,blower_id,site_id,operational_class,max_op_ambient_temp_c,...,casing_temperature_c,vibration_mm_s,daily_degradation,degradation_index,maintenance_event,maintenance_threshold,health_score,maintenance_event_triggered,failure_event,RUL_days
0,2025-01-01,35.777778,23.555556,29.333333,63.9,17.2,ZG150_B001,Site_01,High Duty,48.9,...,50.359494,5.715649,0.007377,0.007377,0,0.6,99.262303,0,0,165.103543
1,2025-01-02,31.777778,20.722222,26.055556,79.5,15.0,ZG150_B001,Site_01,High Duty,48.9,...,41.829335,5.213525,0.005593,0.012970,0,0.6,98.703011,0,0,164.173270
2,2025-01-03,31.611111,22.222222,26.611111,78.3,13.9,ZG150_B001,Site_01,High Duty,48.9,...,41.794693,5.345015,0.005264,0.018233,0,0.6,98.176655,0,0,163.297779
3,2025-01-04,32.722222,23.722222,28.000000,71.4,12.9,ZG150_B001,Site_01,High Duty,48.9,...,46.457210,4.884268,0.005700,0.023934,0,0.6,97.606606,0,0,162.349612
4,2025-01-05,34.111111,22.500000,29.388889,66.3,19.8,ZG150_B001,Site_01,High Duty,48.9,...,49.750704,5.307526,0.006669,0.030603,0,0.6,96.939732,0,0,161.240397


In [ ]:
fleet_df.shape

In [ ]:
maintenance_dates = (
    fleet_df[fleet_df["maintenance_event"] == 1]
    .groupby("blower_id")["date"]
    .min()
    .reset_index()
)

maintenance_dates

In [ ]:
fleet_df.groupby("blower_id")["maintenance_event_triggered"].max()

In [ ]:
fleet_df.groupby("blower_id")["degradation_index"].max().sort_values(ascending=False)

In [ ]:
output_path = SYNTHETIC_DIR / "fleet_rotary_blower_Finaldataset_2025.csv"

fleet_df.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

In [ ]:
#Creating data dictionary

data_dictionary = pd.DataFrame([{
    "column_name":"date",
    "description":"Daily simulation timestamp",
    "unit":"date",
    "data_type":"datetime",
    "source":"Real weather data from Giyani, South Africa for 2025",
    "category":"Temporal"
},
{
    "column_name":"blower_id",
    "description":"Unique identifier for each blower",
    "unit":"integer",
    "data_type":"string",
    "source":"Synthetic configuration",
    "category":"Asset"
},
{
    "column_name":"site_id",
    "description":"Identifier for the site where the blower is located",
    "unit":"integer",
    "data_type":"string",
    "source":"Synthetic configuration",
    "category":"Asset"
},
{
    "column_name":"operational_class",
    "description":"Operational duty classification of the blower)",
    "unit":"N/A",
    "data_type":"string",
    "source":"Synthetic configuration",
    "category":"Operational"
},
{
    "column_name":"daily_op_hours",
    "description":"Number of hours the blower operates each day",
    "unit":"hours/day",
    "data_type":"float",
    "source":"Synthetic configuration",
    "category":"Operational"
},
{
    "column_name":"cumulative_op_hours",
    "description":"Cumulative operating hours of the blower over time",
    "unit":"hours",
    "data_type":"float",
    "source":"Derived from daily_op_hours",
    "category":"Operational"
},
{
    "column_name":"daily_load_percent",
    "description":"Average load on the blower as a percentage of its maximum capacity each day",
    "unit":"%",
    "data_type":"float",
    "source":"Synthetic",
    "category":"Operational"
},
{
    "column_name":"rpm",
    "description":"Rotations per minute of the blower",
    "unit":"rpm",
    "data_type":"float",
    "source":"Derived from daily_load_percent and max_op_rpm",
    "category":"Operational"
},
{
    "column_name":"casing_temperature_c",
    "description":"Temperature of the blower casing in degrees Celsius",
    "unit":"°C",
    "data_type":"float",
    "source":"Synthetic derived",
    "category":"Operational, thermal"
},
{
    "column_name":"dust_index",
    "description":"Synthetic index representing the potential for dust accumulation based on wind gusts and humidity",
    "unit":"0-100",
    "data_type":"float",
    "source":"Synthetic derived",
    "category":"Environmental"
},
{
    "column_name":"max_op_ambient_temp_c",
    "description":"Maximum ambient temperature a blower can operate in, in degrees Celsius",
    "unit":"°C",
    "data_type":"float",
    "source":"Synthetic derived",
    "category":"Operational, thermal"
},
{
    "column_name":"min_op_ambient_temp_c",
    "description":"Minimum ambient temperature a blower can operate in, in degrees Celsius",
    "unit":"°C",
    "data_type":"float",
    "source":"Synthetic derived",
    "category":"Operational, thermal"
},
{
    "column_name":"max_op_discharge_temp_c",
    "description":"Maximum discharge temperature a blower can operate in, in degrees Celsius",
    "unit":"°C",
    "data_type":"float",
    "source":"Synthetic derived",
    "category":"Operational, thermal"
},
{
    "column_name":"amb_temp_avg_c",
    "description":"Average ambient temperature in degrees Celsius",
    "unit":"°C",
    "data_type":"float",
    "source":"Real weather data from Giyani, South Africa for 2025",
    "category":"Environmental"
},
{
    "column_name":"amb_temp_min_c",
    "description":"Minimum ambient temperature in degrees Celsius",
    "unit":"°C",
    "data_type":"float",
    "source":"Real weather data from Giyani, South Africa for 2025",
    "category":"Environmental"
},
{
    "column_name":"amb_temp_max_c",
    "description":"Maximum ambient temperature in degrees Celsius",
    "unit":"°C",
    "data_type":"float",
    "source":"Real weather data from Giyani, South Africa for 2025",
    "category":"Environmental" 
},
{
    "column_name":"humidity",
    "description":"Average ambient humidity as a percentage",
    "unit":"%",
    "data_type":"float",
    "source":"Real weather data from Giyani, South Africa for 2025",
    "category":"Environmental"
},
{
    "column_name":"wind_gust_kph",
    "description":"Maximum wind gust speed in kilometers per hour",
    "unit":"kph",
    "data_type":"float",
    "source":"Real weather data from Giyani, South Africa for 2025",
    "category":"Environmental"
},
{
    "column_name":"vibration_mm_s",
    "description":"Vibration level of the blower in millimeters per second",
    "unit":"mm/s",
    "data_type":"float",
    "source":"Synthetic derived",
    "category":"Mechanical"
},
{
    "column_name":"pressure_diff_psi",
    "description":"Pressure differential across the blower in pounds per square inch",
    "unit":"psi",
    "data_type":"float",
    "source":"Synthetic derived",
    "category":"Mechanical"
},
{
    "column_name":"degradation_index",
    "description":"Cumulative degradation score indicating the health of the blower over time",
    "unit":"N/A",
    "data_type":"float",
    "source":"Derived",
    "category":"Degradation"
},
{
    "column_name":"health_score",
    "description":"Health score of the blower as a percentage (100% = new condition, 0% = failure threshold)",
    "unit":"%",
    "data_type":"float",
    "source":"Derived from degradation index relative to failure threshold",
    "category":"Health Indicator"
},
{
    "column_name":"failure_event",
    "description":"Binary indicator of whether the blower has reached failure threshold (1 = failure, 0 = operational)",
    "unit":"N/A",
    "data_type":"integer",
    "source":"Derived",
    "category":"Target Variable"
},
{
    "column_name":"maintenance_event_triggered",
    "description":"Binary indicator showing whether a maintenance event occurred on that day)",
    "unit":"N/A",
    "data_type":"integer",
    "source":"Derived",
    "category":"Target Variable"
},
{
    "column_name":"maintenance_event",
    "description":"Binary indicator of whether a maintenance event occurred on that day (1 = maintenance occurred, 0 = no maintenance occurred)",
    "unit":"N/A",
    "data_type":"integer",
    "source":"Derived",
    "category":"Target Variable"
},
{
    "column_name":"maintenance_threshold",
    "description":"Dynamic maintenance trigger threshold assigned to each blower unit to simulate variability in maintenance intervention behaviour",
    "unit":"0-1 degradation scale",
    "data_type":"float",
    "source":"Synthetic derived",
    "category":"Maintenance Logic"
},
{
    "column_name":"RUL_days",
    "description":"Estimated remaining useful life of the blower in days until it reaches the failure threshold",
    "unit":"days",
    "data_type":"float",
    "source":"Derived",
    "category":"Predictive Maintenance"
}])

data_dictionary.head()

In [ ]:
data_dictionary.to_csv(
    BASE_DIR / "docs" / "data_dictionary.csv",
    index=False
)